In [1]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score
)

In [2]:
X_train = pd.read_csv("X_train_processed.csv")
X_test = pd.read_csv("X_test_processed.csv")

y_train = pd.read_csv("y_train.csv").values.ravel()
y_test = pd.read_csv("y_test.csv").values.ravel()

X_train.shape, X_test.shape

((3278, 19), (820, 19))

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

le.classes_

array(['At_Risk', 'Malnourished', 'Normal'], dtype=object)

In [4]:
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

In [5]:
xgb.fit(X_train, y_train_enc)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [6]:
y_pred_xgb = xgb.predict(X_test)
y_pred_xgb_labels = le.inverse_transform(y_pred_xgb)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb_labels))
print("\nClassification Report:\n",
      classification_report(y_test, y_pred_xgb_labels))

Accuracy: 0.4475609756097561

Classification Report:
               precision    recall  f1-score   support

     At_Risk       0.27      0.15      0.20       248
Malnourished       0.29      0.08      0.13       166
      Normal       0.50      0.78      0.61       406

    accuracy                           0.45       820
   macro avg       0.35      0.34      0.31       820
weighted avg       0.39      0.45      0.39       820



In [7]:
confusion_matrix(y_test, y_pred_xgb_labels)

array([[ 38,  18, 192],
       [ 26,  14, 126],
       [ 75,  16, 315]])

In [8]:
print("Recall (At_Risk):",
      recall_score(y_test, y_pred_xgb_labels,
                   labels=["At_Risk"], average=None))

print("Recall (Malnourished):",
      recall_score(y_test, y_pred_xgb_labels,
                   labels=["Malnourished"], average=None))

Recall (At_Risk): [0.15322581]
Recall (Malnourished): [0.08433735]


In [11]:
y_proba = xgb.predict_proba(X_test)

In [13]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train_enc)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_enc
)

sample_weights = np.array([class_weights[label] for label in y_train_enc])

xgb_bal = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_bal.fit(X_train, y_train_enc, sample_weight=sample_weights)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [14]:
y_proba = xgb_bal.predict_proba(X_test)
y_proba.shape

(820, 3)

In [15]:
le.classes_

array(['At_Risk', 'Malnourished', 'Normal'], dtype=object)

In [16]:
pd.DataFrame(
    y_proba[:10],
    columns=le.classes_
)

,At_Risk,Malnourished,Normal
0,0.472531,0.267561,0.259908
1,0.322323,0.282000,0.395676
2,0.138487,0.614321,0.247191
3,0.272704,0.144007,0.583289
4,0.293894,0.186196,0.519910
5,0.257291,0.242230,0.500479
6,0.126005,0.539737,0.334258
7,0.493376,0.220762,0.285862
8,0.211691,0.254764,0.533546
9,0.323973,0.380166,0.295861


In [17]:
threshold = 0.20

y_pred_threshold = []

for probs in y_proba:
    if probs[1] >= threshold:   # Malnourished
        y_pred_threshold.append("Malnourished")
    elif probs[0] >= 0.33:      # At_Risk (optional secondary rule)
        y_pred_threshold.append("At_Risk")
    else:
        y_pred_threshold.append("Normal")

y_pred_threshold = np.array(y_pred_threshold)


In [18]:
from sklearn.metrics import classification_report, confusion_matrix, recall_score

print(classification_report(y_test, y_pred_threshold))
confusion_matrix(y_test, y_pred_threshold)


              precision    recall  f1-score   support

     At_Risk       0.24      0.12      0.17       248
Malnourished       0.20      0.75      0.31       166
      Normal       0.36      0.06      0.10       406

    accuracy                           0.22       820
   macro avg       0.27      0.31      0.19       820
weighted avg       0.29      0.22      0.16       820



array([[ 31, 192,  25],
       [ 25, 124,  17],
       [ 71, 311,  24]])

In [19]:
print("Recall (At_Risk):",
      recall_score(y_test, y_pred_threshold,
                   labels=["At_Risk"], average=None))

print("Recall (Malnourished):",
      recall_score(y_test, y_pred_threshold,
                   labels=["Malnourished"], average=None))

Recall (At_Risk): [0.125]
Recall (Malnourished): [0.74698795]


In [20]:
for t in [0.10, 0.15, 0.20, 0.25, 0.30]:
    y_tmp = []
    for probs in y_proba:
        if probs[1] >= t:
            y_tmp.append("Malnourished")
        else:
            y_tmp.append("Normal")
    y_tmp = np.array(y_tmp)

    rec = recall_score(y_test, y_tmp,
                       labels=["Malnourished"], average=None)[0]
    print(f"Threshold {t:.2f} → Malnourished Recall: {rec:.3f}")

Threshold 0.10 → Malnourished Recall: 0.994
Threshold 0.15 → Malnourished Recall: 0.873
Threshold 0.20 → Malnourished Recall: 0.747
Threshold 0.25 → Malnourished Recall: 0.614
Threshold 0.30 → Malnourished Recall: 0.488
